# 05 — Export Everything the Laptop Needs

This notebook exports the processed tables from Unity Catalog / Delta into parquet files in `/Volumes/workspace/m5/exports` so they can be downloaded and used locally for LightGBM training, backtesting, and evaluation.

### Exported Tables:
1. **`features`**: Filtered for `d_num >= TRAIN_START_D` (d >= 1000) to keep memory lightweight, repartitioned by `store_id`.
2. **`baselines`**: Filtered to exclude `'TUNE'` fold (`fold != 'TUNE'`), containing SeasonalNaive, AutoETS, and Croston predictions.
3. **`series_stats`**: Per-series scale and weights per fold computed in Step 7.
4. **`series_profile`**: Adi, cv2, and demand categorization computed in Step 2/7.

> **Why `repartition("store_id")` here?**
> In Step 7.2 (computation), 10 partitions would have starved cluster cores and created a massive bottleneck.
> Here in Step 8.3 (writing to disk), one writer per store creates exactly one clean directory per store with a few large, compressed parquet files instead of thousands of tiny fragmented files.

In [0]:
import sys
REPO = "/Workspace/Users/veysel/m5-forecasting-spark"  # adjust if your repo path differs
if REPO not in sys.path:
    sys.path.append(REPO)

from pyspark.sql import functions as F
from src.config import TRAIN_START_D

EXP = "/Volumes/workspace/m5/exports"
print(f"Export destination: {EXP}")
print(f"Training start day cutoff: d_num >= {TRAIN_START_D}")

### 1. Export `features` Table

In [0]:
print("Exporting features...")
(
    spark.table("workspace.m5.features")
    .filter(F.col("d_num") >= TRAIN_START_D)
    .repartition("store_id")
    .write.mode("overwrite")
    .partitionBy("store_id")
    .parquet(f"{EXP}/features")
)
print("Features export complete.")

### 2. Export `baselines` Table (F1–F3, excluding TUNE)

In [0]:
print("Exporting baselines...")
(
    spark.table("workspace.m5.baselines")
    .filter("fold != 'TUNE'")
    .write.mode("overwrite")
    .parquet(f"{EXP}/baselines")
)
print("Baselines export complete.")

### 3. Export `series_stats` and `series_profile`

In [0]:
print("Exporting series_stats...")
spark.table("workspace.m5.series_stats").write.mode("overwrite").parquet(f"{EXP}/series_stats")

print("Exporting series_profile...")
spark.table("workspace.m5.series_profile").write.mode("overwrite").parquet(f"{EXP}/series_profile")

print("All exports completed successfully!")

### 4. Verify Export Files in Volume

In [0]:
display(dbutils.fs.ls(EXP))